# E-commerce Text Mining & Sales Trend Analysis

This project applies Python-based text mining and data analysis to e-commerce product and customer-comment data.

## Objectives
- Clean and preprocess customer comments
- Identify comments related to product names using Chinese word segmentation
- Analyze comment sentiment with SnowNLP
- Aggregate product sales by category and month
- Visualize category-level sales trends over time

## Technologies
Python · Pandas · Jieba · SnowNLP · Matplotlib · Jupyter Notebook

> **Data note:** The original course-project datasets are not included in this repository. The notebook therefore uses configurable local paths for the datasets. No original user IDs or raw customer comments are published here.


## 1. Setup


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import jieba
import jieba.posseg as pseg
from snownlp import SnowNLP


## 2. Load data

Place the two comment/product CSV files in `data/` and update the paths if needed. The monthly product Excel files can be placed in `data/monthly/`.


In [ ]:
from pathlib import Path

DATA_DIR = Path("data")
MONTHLY_DIR = DATA_DIR / "monthly"

merged_df = pd.read_csv(DATA_DIR / "merged_df.csv")
TRBOX_comment_df = pd.read_csv(DATA_DIR / "TRBOX_comment_df.csv")

print(f"Product records: {len(merged_df):,}")
print(f"Comment records: {len(TRBOX_comment_df):,}")


## 3. Comment preprocessing

The original project removed promotional/advertising comments before matching comments to product names. Specific source user IDs from the original dataset are intentionally not included in this public version.


In [ ]:
# Remove promotional / advertising keywords
keywords_to_remove = [
    "百標", "清倉", "新氣象新活動", "關鍵字",
    "十萬讚好開心", "好買", "火力全開"
]

pattern = "|".join(keywords_to_remove)
TRBOX_comment_df = TRBOX_comment_df[
    ~TRBOX_comment_df["MESSAGE"].fillna("").str.contains(pattern, na=False)
].copy()

print(f"Comments after preprocessing: {len(TRBOX_comment_df):,}")


## 4. Extract product-related terms

Jieba POS tagging is used to extract noun candidates from product names.


In [ ]:
def extract_nouns(sentence):
    words = pseg.cut(str(sentence))
    return [word for word, flag in words if flag.startswith("n")]

segmented_products = []
for name in merged_df["name"].dropna():
    segmented_products.extend(extract_nouns(name))

# Remove one-character and generic terms that are not useful product indicators.
segmented_products = [word for word in segmented_products if len(word) > 1]
words_to_remove = {"顏色", "大家", "兄弟", "公分"}
segmented_products = [
    word for word in segmented_products if word not in words_to_remove
]

# Longer terms are matched first to reduce accidental short-word matches.
segmented_products = sorted(set(segmented_products), key=len, reverse=True)

print(f"Product-related terms: {len(segmented_products):,}")
print(segmented_products[:30])


## 5. Match product-related comments


In [ ]:
related_comment = []
related_nouns = []

for comment in TRBOX_comment_df["MESSAGE"].fillna(""):
    for product in segmented_products:
        if product in comment:
            related_comment.append(comment)
            related_nouns.append(product)
            break

useful_df = pd.DataFrame({
    "related_comment": related_comment,
    "related_nouns": related_nouns
})

print(f"Matched comments: {len(useful_df):,}")
useful_df.head()


## 6. Sentiment analysis

SnowNLP is used to generate a sentiment score for each matched comment. A score above 0.5 is classified as positive in the original analysis.


In [ ]:
useful_df["sentiment_score"] = useful_df["related_comment"].apply(
    lambda comment: SnowNLP(comment).sentiments
)
useful_df["sentiment"] = np.where(
    useful_df["sentiment_score"] > 0.5, "Positive", "Negative"
)

useful_df.head()


## 7. Monthly sales trend analysis

The original project combined monthly product Excel files and aggregated total quantity by product category and month.


In [ ]:
monthly_files = sorted(MONTHLY_DIR.glob("商品資料-*.xlsx"))

if not monthly_files:
    raise FileNotFoundError(
        "No monthly Excel files found. Place the original monthly product files in data/monthly/."
    )

all_commodity_df = pd.concat(
    [pd.read_excel(file) for file in monthly_files],
    ignore_index=True
)

all_commodity_df = all_commodity_df[["time", "category", "total_quantity"]].copy()
all_commodity_df["time"] = pd.to_datetime(all_commodity_df["time"])

trend_result_df = (
    all_commodity_df
    .groupby(["category", all_commodity_df["time"].dt.to_period("M")])["total_quantity"]
    .sum()
    .reset_index()
)

trend_result_df = trend_result_df[trend_result_df["time"].dt.year != 2021].copy()
trend_result_df["time"] = trend_result_df["time"].dt.to_timestamp()

trend_result_df.head()


## 8. Visualize category trends


In [ ]:
def plot_category_trends(df, categories, title):
    plot_df = df[df["category"].fillna("").str.contains("|".join(categories), na=False)].copy()

    plt.figure(figsize=(10, 6))
    for category in categories:
        category_data = plot_df[
            plot_df["category"].fillna("").str.contains(category, na=False)
        ]
        plt.plot(
            category_data["time"],
            category_data["total_quantity"],
            label=category
        )

    plt.xlabel("Time")
    plt.ylabel("Total Quantity")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()


top_categories = [
    "居家生活", "美妝保健", "美食、伴手禮", "嬰幼童與母親", "飾品、配件"
]

plot_category_trends(
    trend_result_df,
    top_categories,
    "Category-wise Total Quantity Over Time"
)


In [ ]:
other_categories = [
    "運動、健身", "男女鞋", "女生包包、精品", "3C與筆電", "電玩、遊戲"
]

plot_category_trends(
    trend_result_df,
    other_categories,
    "Category-wise Total Quantity Over Time"
)


## Key takeaways

- Python and Pandas were used for data cleaning, transformation, aggregation, and visualization.
- Jieba was used for Chinese word segmentation and noun extraction from product names.
- SnowNLP was used for rule-based sentiment scoring of product-related comments.
- Monthly sales data was aggregated by product category to explore changes over time.

## Reproducibility

The original datasets are omitted from this public repository. To run the notebook, provide the corresponding datasets locally under `data/` and `data/monthly/`.
